In [1]:
import sys
import os

# go one level up from notebooks → project root
repo_path = os.path.abspath('..')

sys.path.append(repo_path)

print(repo_path)


c:\Users\hp\NLP-sequence-classification


In [2]:
from src.data_loader import get_BESSTIE_splits
import pandas as pd
from src.lr_feature_extraction import tfidf_features, load_tfidf_features
from models.svm_tfidf import MultiOutputSVM, SeparateSVM

c:\Users\hp\NLP-sequence-classification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# loading and getting the splits from the dataset
df_all, df_train, df_validation, df_test = get_BESSTIE_splits()

In [4]:
# Extract TFIDF Features
X_train, X_validation, X_test, vectorizer = tfidf_features(
    df_train, df_validation, df_test,
    text_column='text',
    max_features=15000,
    save_path="./models/tfidf"
)

## Multi-Output SVM
Single model predicting both Sarcasm and Sentiment simultaneously

In [5]:
# train the multi-output model
multi_svm = MultiOutputSVM()
multi_svm.train_MultiOutputSVM(X_train, df_train)

In [6]:
# Evaluate the multi-output model
multi_svm_results = multi_svm.MultiOutputSVM_evaluation(X_validation, df_validation)

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {multi_svm_results['Sarcasm']['Accuracy']:.4f}")
print(f"   Precision: {multi_svm_results['Sarcasm']['Precision']:.4f}")
print(f"   Recall:    {multi_svm_results['Sarcasm']['Recall']:.4f}")
print(f"   F1-Score:  {multi_svm_results['Sarcasm']['F1']:.4f}")
print(f"   F1-Macro:  {multi_svm_results['Sarcasm']['F1_Macro']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {multi_svm_results['Sentiment']['Accuracy']:.4f}")
print(f"   Precision: {multi_svm_results['Sentiment']['Precision']:.4f}")
print(f"   Recall:    {multi_svm_results['Sentiment']['Recall']:.4f}")
print(f"   F1-Score:  {multi_svm_results['Sentiment']['F1']:.4f}")
print(f"   F1-Macro:  {multi_svm_results['Sentiment']['F1_Macro']:.4f}")

Sarcasm labels - unique: [0 1] mean: 0.14057507987220447
Sentiment labels - unique: [0 1] mean: 0.48881789137380194
Sarcasm predictions - unique: [0 1] mean: 0.14376996805111822
Sentiment predictions - unique: [0 1] mean: 0.5239616613418531

📌 SARCASM DETECTION:
   Accuracy:  0.8051
   Precision: 0.3111
   Recall:    0.3182
   F1-Score:  0.3146
   F1-Macro:  0.6005

📌 SENTIMENT ANALYSIS:
   Accuracy:  0.8243
   Precision: 0.7988
   Recall:    0.8562
   F1-Score:  0.8265
   F1-Macro:  0.8243


In [7]:
# Save multi-output model
multi_svm.save_MultiOutputSVM_model("./models/multi_output_svm.pkl")

✅ Model saved to ./models/multi_output_svm.pkl


## Separate SVM Models — BESSTIE Paper Approach
One model for Sarcasm, one model for Sentiment

In [8]:
# train separate models
separate_svm = SeparateSVM()
separate_svm.train_SeparateSVM(X_train, df_train)

In [9]:
# Evaluate separate models
separate_svm_results = separate_svm.SeparateSVM_evaluation(X_validation, df_validation)

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {separate_svm_results['Sarcasm']['Accuracy']:.4f}")
print(f"   Precision: {separate_svm_results['Sarcasm']['Precision']:.4f}")
print(f"   Recall:    {separate_svm_results['Sarcasm']['Recall']:.4f}")
print(f"   F1-Score:  {separate_svm_results['Sarcasm']['F1']:.4f}")
print(f"   F1-Macro:  {separate_svm_results['Sarcasm']['F1_Macro']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {separate_svm_results['Sentiment']['Accuracy']:.4f}")
print(f"   Precision: {separate_svm_results['Sentiment']['Precision']:.4f}")
print(f"   Recall:    {separate_svm_results['Sentiment']['Recall']:.4f}")
print(f"   F1-Score:  {separate_svm_results['Sentiment']['F1']:.4f}")
print(f"   F1-Macro:  {separate_svm_results['Sentiment']['F1_Macro']:.4f}")

Sarcasm labels - unique: [0 1] mean: 0.14057507987220447
Sentiment labels - unique: [0 1] mean: 0.48881789137380194
Sarcasm predictions - unique: [0 1] mean: 0.14376996805111822
Sentiment predictions - unique: [0 1] mean: 0.5239616613418531

📌 SARCASM DETECTION:
   Accuracy:  0.8051
   Precision: 0.3111
   Recall:    0.3182
   F1-Score:  0.3146
   F1-Macro:  0.6005

📌 SENTIMENT ANALYSIS:
   Accuracy:  0.8243
   Precision: 0.7988
   Recall:    0.8562
   F1-Score:  0.8265
   F1-Macro:  0.8243


In [10]:
# Save separate models
separate_svm.save_SeparateSVM_models("./models")

✅ Models saved to ./models/


## FINAL TEST EVALUATION

In [11]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

test_predictions = separate_svm.prediction_SeparateSVM(X_test)

sarcasm_true_labels = df_test['Sarcasm'].astype(int).values
sentiment_true_labels = df_test['Sentiment'].astype(int).values

sarcasm_f1        = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_f1_macro  = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'], average='macro')
sarcasm_precision = precision_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_recall    = recall_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_accuracy  = accuracy_score(sarcasm_true_labels, test_predictions['Sarcasm'])

sentiment_f1        = f1_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_f1_macro  = f1_score(sentiment_true_labels, test_predictions['Sentiment'], average='macro')
sentiment_precision = precision_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_recall    = recall_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_accuracy  = accuracy_score(sentiment_true_labels, test_predictions['Sentiment'])

print("\n📌 SARCASM DETECTION (Test):")
print(f"   Accuracy:  {sarcasm_accuracy:.4f}")
print(f"   Precision: {sarcasm_precision:.4f}")
print(f"   Recall:    {sarcasm_recall:.4f}")
print(f"   F1-Score:  {sarcasm_f1:.4f}")
print(f"   F1-Macro:  {sarcasm_f1_macro:.4f}")

print("\n📌 SENTIMENT ANALYSIS (Test):")
print(f"   Accuracy:  {sentiment_accuracy:.4f}")
print(f"   Precision: {sentiment_precision:.4f}")
print(f"   Recall:    {sentiment_recall:.4f}")
print(f"   F1-Score:  {sentiment_f1:.4f}")
print(f"   F1-Macro:  {sentiment_f1_macro:.4f}")


📌 SARCASM DETECTION (Test):
   Accuracy:  0.8172
   Precision: 0.3494
   Recall:    0.3574
   F1-Score:  0.3533
   F1-Macro:  0.6234

📌 SENTIMENT ANALYSIS (Test):
   Accuracy:  0.8209
   Precision: 0.8193
   Recall:    0.8124
   F1-Score:  0.8158
   F1-Macro:  0.8208
